[Reference](https://jupyter2607.medium.com/learn-to-deploy-llm-apps-run-ollama-on-google-colab-and-connect-locally-with-ngrok-4b84a1106270)

# Step 1: Install Dependencies

In [1]:
!pip install pyngrok
!curl https://ollama.ai/install.sh | sh

  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 13281    0 13281    0     0  38092      0 --:--:-- --:--:-- --:--:-- 38163
>>> Installing ollama to /usr/local
>>> Downloading Linux amd64 bundle
######################################################################## 100.0%
>>> Creating ollama user...
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.


# Step 2: Set Up Ngrok

In [2]:
!wget https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
!tar xvzf ngrok-v3-stable-linux-amd64.tgz

--2025-10-19 13:02:14--  https://bin.equinox.io/c/bNyj1mQVY4c/ngrok-v3-stable-linux-amd64.tgz
Resolving bin.equinox.io (bin.equinox.io)... 13.248.244.96, 35.71.179.82, 99.83.220.108, ...
Connecting to bin.equinox.io (bin.equinox.io)|13.248.244.96|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 9315483 (8.9M) [application/octet-stream]
Saving to: ‘ngrok-v3-stable-linux-amd64.tgz’

ngrok-v3-stable-lin 100%[===================>]   8.88M  2.78MB/s    in 3.2s    

2025-10-19 13:02:19 (2.78 MB/s) - ‘ngrok-v3-stable-linux-amd64.tgz’ saved [9315483/9315483]

ngrok


# Step 3: Authenticate Ngrok
Next, you go to https://ngrok.com/ create a Ngrok account. After logging in, go to https://dashboard.ngrok.com/get-started/your-authtoken to create your Authtoken.

In [3]:
from google.colab import userdata
auth_token = userdata.get('NGROK_AUTH_TOKEN')
!./ngrok authtoken {auth_token}

# Step 4: Start Ollama and Expose the API


In [4]:
!ollama serve & ./ngrok http 11434 --host-header="localhost:11434" --log stdout

In [5]:
!pip install colab-xterm
%load_ext colabxterm

In [6]:
%xterm

```
!ollama serve & ./ngrok http 11434 --url=[YOUR_DOMAIN_URL] --host-header="localhost:11434" --log stdout & sleep 5s && ollama pull qwen3
```

# Step 5: Connect Ollama server on your machine


In [7]:
!curl https://ollama.ai/install.sh | sh
# !sudo apt update
# !sudo apt install curl

In [8]:
ollama

In [9]:
export OLLAMA_HOST=https://e8c9-35-231-95-189.ngrok-free.app

In [10]:
ollama list

In [11]:
ollama pull llama3.1

# Step 6: Create your LLM project on your machine

In [12]:
import nest_asyncio
nest_asyncio.apply()

from pydantic import BaseModel

from pydantic_ai import Agent
from pydantic_ai.models.openai import OpenAIModel
from pydantic_ai.providers.openai import OpenAIProvider


class CityLocation(BaseModel):
    city: str
    country: str


ollama_model = OpenAIModel(
    model_name='llama3.1',
    provider=OpenAIProvider(base_url='https://47e0-35-204-248-251.ngrok-free.app/v1')
)
agent = Agent(ollama_model, output_type=CityLocation)

result = agent.run_sync('Where were the olympics held in 2012?')
print(result.output)
#> city='London' country='United Kingdom'
print(result.usage())

In [13]:
import requests

# Note the url would be: /v1/chat/completions
url = "https://e8c9-35-231-95-189.ngrok-free.app/v1/chat/completions"
headers = {
    "Content-Type": "application/json"
}

data = {
    "model": "llama3.1",
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hello!"}
    ]
}

response = requests.post(url, headers=headers, json=data)

# Print the full JSON response
print(response.json())
{'id': 'chatcmpl-425', 'object': 'chat.completion', 'created': 1748171837, 'model': 'llama3.1', 'system_fingerprint': 'fp_ollama', 'choices': [{'index': 0, 'message': {'role': 'assistant', 'content': "How can I assist you today? Do you have a specific problem or question you'd like help with, or would you just like to chat?"}, 'finish_reason': 'stop'}], 'usage': {'prompt_tokens': 23, 'completion_tokens': 30, 'total_tokens': 53}}
# OR print just the assistant's reply
print(response.json()['choices'][0]['message']['content'])
# > How can I assist you today? Do you have a specific problem or question you'd like help with, or would you just like to chat?